# Reddit (perceived safety, Y2): June–August 2026

Read from the published files in `Data/reddit_2026-06-01_2026-08-31/`; nothing is recomputed or written.

1. What was collected
2. Areas: mentions and the share of negative records, with 95% confidence intervals
3. Environmental-cue precision
4. How the headline shares move under four sentiment scorers
5. The pipeline's self-checks

Only counts and rates are shown, never the text of a post or comment (`Docs/data_ethics.md`).

In [ ]:
# Shared folders, project modules and plot settings
from _setup import *
# Show plots inside the notebook
%matplotlib inline

## Step 1: What was collected

Comments written after 31 August stay in the files but are flagged (`comment_in_window`) and left out of every table.

In [ ]:
# The raw posts and comments, as the collector saved them
raw = REDDIT / "01_raw_collection"
posts = pd.read_csv(next(raw.glob("saskatoon_safety_posts_*.csv")))
comments = pd.read_csv(next(raw.glob("saskatoon_safety_comments_*.csv")))
# Comments written inside the study window
in_window = int(comments["comment_in_window"].astype(bool).sum())
# The four counts in one small table
pd.Series({"posts": len(posts), "comments": len(comments), "comments inside 1 Jun - 31 Aug": in_window,
           "comments written after 31 Aug": len(comments) - in_window}, name="count").to_frame()

## Step 2: Areas

Records that mention each area, how many threads they come from, and the share scored negative by adjusted VADER. The 95% intervals come from resampling whole threads. Areas with fewer than 15 mentions have very wide intervals and are left out here.

In [ ]:
# One row per area
areas = pd.read_csv(REDDIT / "02_derived_analysis" / "neighbourhood_summary.csv")
# Areas with at least 15 mentions, most mentioned first
big = areas[areas["total_mentions"] >= 15].sort_values("total_mentions", ascending=False)
big[["neighbourhood", "total_mentions", "n_threads", "pct_negative", "pct_neg_ci95_low", "pct_neg_ci95_high"]]

In [ ]:
# How far each interval reaches below and above its share
below = big["pct_negative"] - big["pct_neg_ci95_low"]
above = big["pct_neg_ci95_high"] - big["pct_negative"]
# One bar per area, with its interval
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(big["neighbourhood"], big["pct_negative"], xerr=[below, above], color="#2a78d6", capsize=3)
# Most mentioned area at the top
ax.invert_yaxis()
ax.set_xlabel("% of records negative (adjusted VADER), 95% CI")
ax.set_title("Areas with at least 15 mentions, June-August 2026", loc="left")
plt.show()

## Step 3: Environmental-cue precision

For each cue, 30 matches from June records were judged: does the matched word really describe the cue? These judgments are an automated first pass that has not yet been checked by hand (`Scripts/reddit_analysis/scripts/blind_review.py` is that step), so read them as provisional. Lighting is the weak cue: treat lighting counts as an upper bound.

In [ ]:
# Precision of each cue with its 95% CI
precision = pd.read_csv(REDDIT / "05_audits" / "env_cue_precision_heldout.csv")
precision[["cue", "n_judged", "n_correct", "precision", "ci95_low", "ci95_high"]]

## Step 4: Sentiment sensitivity

The same shares under four ways of scoring sentiment. Adjusted VADER (`B_adjusted`) is the one used everywhere else.

In [ ]:
# Share of negative comments overall, Downtown and Riverbank under each scorer
sensitivity = pd.read_csv(REDDIT / "05_audits" / "sentiment_sensitivity.csv")
sensitivity[["spec", "comments_pct_negative", "downtown_cbd_pct_negative", "riverbank_meewasin_pct_negative"]]

## Step 5: The pipeline's self-checks

In [ ]:
# The verification report: one line per check, marked [PASS] or [FAIL]
report = (REDDIT / "05_audits" / "verification_report.txt").read_text()
# How many passed and how many failed
passed = report.count("[PASS]")
failed = report.count("[FAIL]")
print(f"{passed} checks pass, {failed} fail")

## Takeaway

- These are the published numbers; `Scripts/reddit_analysis/run_all.sh` produces them and the flight checks
  that they rebuild exactly.
- Compare areas only where the intervals don't overlap, and keep the cue precision provisional until it has
  been checked by hand.